In [1]:
### Video Understanding Models (Late Fusion, Early Fusion, 3D CNN) Comparison


import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

## Define tensor shape and number of classes
N, T, C, H, W = 2, 8, 3, 64, 64
NUM_CLASSES = 5

## Generate random video tensors and labels
x = torch.randn(N, T, C, H, W)
target = torch.randint(0, NUM_CLASSES, (N,))


## Define LateFusion model with a per-frame encoder
class LateFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.frame_encoder = nn.Sequential(
            nn.Conv2d(C, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Linear(16, NUM_CLASSES)

    def forward(self, x):
        n, t, c, h, w = x.shape

        # Treat each frame as an independent image
        z = x.reshape(n * t, c, h, w)
        z = self.frame_encoder(z).flatten(1)

        # Generate clip features by averaging frame features
        z = z.reshape(n, t, -1).mean(dim=1)
        return self.classifier(z)


## Define EarlyFusion model by merging Time and Channel dimensions
class EarlyFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(C * T, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(16, NUM_CLASSES)
        )

    def forward(self, x):
        n, t, c, h, w = x.shape

        # Combine the time axis with the channel dimensions: [N, C*T, H, W]
        z = x.permute(0, 2, 1, 3, 4)
        z = z.reshape(n, c * t, h, w)
        return self.model(z)


## Define 3D CNN model using 3D Convolution over Time, Height, and Width
class Simple3DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv3d(
                C, 16,
                kernel_size=(3, 3, 3),
                padding=(1, 1, 1)
            ),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(16, NUM_CLASSES)
        )

    def forward(self, x):
        # [N,T,C,H,W] → [N,C,T,H,W]
        z = x.permute(0, 2, 1, 3, 4)
        return self.model(z)


models = {
    "late_fusion": LateFusion(),
    "early_fusion": EarlyFusion(),
    "conv3d": Simple3DCNN()
}


## Model Benchmarking: Dimensions, Parameters, and Cross-Entropy Loss
for name, model in models.items():
    model.zero_grad()

    logits = model(x)
    loss = F.cross_entropy(logits, target)
    loss.backward()

    params = sum(p.numel() for p in model.parameters())

    print(
        f"{name:13s}",
        "logits:", tuple(logits.shape),
        "params:", params,
        "loss:", round(loss.item(), 4)
    )

late_fusion   logits: (2, 5) params: 533 loss: 1.6962
early_fusion  logits: (2, 5) params: 3557 loss: 1.8403
conv3d        logits: (2, 5) params: 1397 loss: 1.5691
